# PoliMillionaire Speech Final Runs

Same model harness as the text notebook. The only change is that the live game uses the speech API and local Whisper transcription.


## Setup

In [1]:
import gc
import importlib.util
import os
import sys
import time
from datetime import datetime, timezone
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "polimillionaire" / "__init__.py").exists():
            return candidate
    raise RuntimeError("Open this notebook from inside the project folder.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
for path in [REPO_ROOT / "src", REPO_ROOT]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

os.environ.setdefault("HF_HOME", str(REPO_ROOT / "data" / "hf_home"))
os.environ.setdefault("HUGGINGFACE_HUB_CACHE", str(REPO_ROOT / "data" / "hf_home" / "hub"))
print("Repo:", REPO_ROOT)
print("HF cache:", os.environ["HF_HOME"])

Repo: C:\Users\sjuan\Desktop\NLP\project\nlp-polimillionaire
HF cache: C:\Users\sjuan\Desktop\NLP\project\nlp-polimillionaire\data\hf_home


## Speech Run Settings


In [2]:
RUN_API_CHECK = False
RUN_LIVE_GAME = True
RUN_OFFLINE_BENCHMARK = False
BLOCK_LIVE_ON_BENCHMARK_FAILURE = True
PRELOAD_SELECTED_STRATEGIES = True
CLEAR_MEMORY_AFTER_EACH_MODEL = True
PROMPT_FOR_CREDENTIALS = False

RUN_BEST_SINGLE_ALL_CATEGORIES = True
RUN_BEST_BY_CATEGORY = True

API_URL = "http://131.175.15.22:51111/"
COMPETITION_IDS = [0, 1, 2, 3, 4, 5]
SAFE_DELAY_SECONDS = 2.0
ANSWER_TIMEOUT_SECONDS = 25.0
SPEECH_AUDIO_FETCH_DELAY_SECONDS = 0.2
SAVE_SPEECH_AUDIO = True
SPEECH_AUDIO_ROOT = REPO_ROOT / "results" / "audio"
SPEECH_WHISPER_BACKEND = "transformers_whisper"
SPEECH_WHISPER_MODEL_ID = "openai/whisper-base.en"
SPEECH_WHISPER_DEVICE = "cpu"
SPEECH_WHISPER_DTYPE = "float32"
SPEECH_WHISPER_FALLBACK_BACKEND = "faster_whisper"
SPEECH_WHISPER_FALLBACK_MODEL_ID = "distil-large-v3"
SPEECH_WHISPER_FALLBACK_DEVICE = "cpu"
SPEECH_WHISPER_FALLBACK_DTYPE = "float32"
SPEECH_WHISPER_FALLBACK_COMPUTE_TYPE = "int8"
RUN_SPEECH_REPLAY_BENCHMARK = True
BLOCK_LIVE_ON_SPEECH_REPLAY_FAILURE = True
SPEECH_REPLAY_MAX_AUDIO_SETS = 2
SPEECH_REPLAY_MAX_SECONDS_PER_SET = 25.0

CATEGORY_NAMES = {
    0: "entertainment",
    1: "history",
    2: "science",
    3: "math",
    4: "philosophy_psychology",
    5: "news",
}

ARCHITECTURES = {
    "gemma_e2b_two_agent_quant_council": {
        "label": "Gemma E2B two-agent quantized council + quantized judge",
        "short_name": "gemma_e2b_two_agent",
        "kind": "gemma_tool_council_quant",
        "model_id": "google/gemma-4-E2B-it",
    },
    "qwen_two_agent_quant_council": {
        "label": "Qwen 3.5 2B two-agent quantized council + quantized judge",
        "short_name": "qwen_two_agent",
        "kind": "qwen_tool_council_quant",
        "model_id": "Qwen/Qwen3.5-2B",
    },
    "mixed_gemma_qwen_routed_rag": {
        "label": "Gemma + Qwen 4-bit mixed routed RAG",
        "short_name": "mixed_gemma_qwen",
        "kind": "mixed_quantized_rag",
        "model_id": "google/gemma-4-E2B-it + Qwen/Qwen3.5-2B",
    },
    "gemma_e2b_routed_rag": {
        "label": "Gemma E2B 4-bit tools + routed RAG",
        "short_name": "gemma_e2b_routed_rag",
        "kind": "gemma_tool_rag_quant",
        "model_id": "google/gemma-4-E2B-it",
    },
}

BEST_SINGLE_ARCHITECTURE_KEY = "gemma_e2b_two_agent_quant_council"
BEST_BY_CATEGORY_KEYS = {
    0: "gemma_e2b_two_agent_quant_council",
    1: "gemma_e2b_two_agent_quant_council",
    2: "gemma_e2b_two_agent_quant_council",
    3: "qwen_two_agent_quant_council",
    4: "gemma_e2b_routed_rag",
    5: "gemma_e2b_two_agent_quant_council",
}

print("API check:", RUN_API_CHECK, "Live game:", RUN_LIVE_GAME, "Benchmark:", RUN_OFFLINE_BENCHMARK)
print("Run global best:", RUN_BEST_SINGLE_ALL_CATEGORIES, "Run category best:", RUN_BEST_BY_CATEGORY)
print("Block on benchmark failure:", BLOCK_LIVE_ON_BENCHMARK_FAILURE)
print("Competition IDs:", COMPETITION_IDS)
print("Speech API mode: True")
print("Save speech audio:", SAVE_SPEECH_AUDIO, "Audio fetch delay:", SPEECH_AUDIO_FETCH_DELAY_SECONDS)
print("Whisper:", SPEECH_WHISPER_BACKEND, SPEECH_WHISPER_MODEL_ID, "device:", SPEECH_WHISPER_DEVICE, "dtype:", SPEECH_WHISPER_DTYPE)
print("Whisper fallback:", SPEECH_WHISPER_FALLBACK_BACKEND, SPEECH_WHISPER_FALLBACK_MODEL_ID, "device:", SPEECH_WHISPER_FALLBACK_DEVICE, "compute:", SPEECH_WHISPER_FALLBACK_COMPUTE_TYPE)
print("Speech replay benchmark:", RUN_SPEECH_REPLAY_BENCHMARK, "block:", BLOCK_LIVE_ON_SPEECH_REPLAY_FAILURE)
print("Global best:", ARCHITECTURES[BEST_SINGLE_ARCHITECTURE_KEY]["label"])
print("Best by category:")
for competition_id in COMPETITION_IDS:
    key = BEST_BY_CATEGORY_KEYS[competition_id]
    print("-", competition_id, CATEGORY_NAMES[competition_id], "->", ARCHITECTURES[key]["label"])


API check: False Live game: True Benchmark: False
Run global best: True Run category best: True
Block on benchmark failure: True
Competition IDs: [0, 1, 2, 3, 4, 5]
Speech API mode: True
Save speech audio: True Audio fetch delay: 0.2
Whisper: transformers_whisper openai/whisper-base.en device: cpu dtype: float32
Whisper fallback: faster_whisper distil-large-v3 device: cpu compute: int8
Speech replay benchmark: True block: True
Global best: Gemma E2B two-agent quantized council + quantized judge
Best by category:
- 0 entertainment -> Gemma E2B two-agent quantized council + quantized judge
- 1 history -> Gemma E2B two-agent quantized council + quantized judge
- 2 science -> Gemma E2B two-agent quantized council + quantized judge
- 3 math -> Qwen 3.5 2B two-agent quantized council + quantized judge
- 4 philosophy_psychology -> Gemma E2B 4-bit tools + routed RAG
- 5 news -> Gemma E2B two-agent quantized council + quantized judge


## Login

In [3]:
import getpass

from dotenv import load_dotenv
from millionaire_client import AuthenticationError, MillionaireClient

# Load environment variables from .env file
try:
    load_dotenv()
except Exception as e:
    print(f"Error loading .env file: {e}"   )

USERNAME = os.environ.get("POLIMILLIONAIRE_USERNAME")
PASSWORD = os.environ.get("POLIMILLIONAIRE_PASSWORD")
try:
    from google.colab import userdata
    USERNAME = USERNAME or userdata.get("POLIMILLIONAIRE_USERNAME")
    PASSWORD = PASSWORD or userdata.get("POLIMILLIONAIRE_PASSWORD")
except Exception:
    pass
if PROMPT_FOR_CREDENTIALS and not USERNAME:
    USERNAME = input("PoliMillionaire username: ").strip()
if PROMPT_FOR_CREDENTIALS and not PASSWORD:
    PASSWORD = getpass.getpass("PoliMillionaire password: ")
print("Username configured:", bool(USERNAME))
print("Password configured:", bool(PASSWORD))

Username configured: True
Password configured: True


## API Check


In [4]:
client = None
if RUN_API_CHECK:
    if not USERNAME or not PASSWORD:
        raise RuntimeError("Set credentials first.")
    try:
        client = MillionaireClient(API_URL)
        user = client.login(USERNAME, PASSWORD)
        print("Logged in as", user.username)
        for competition in client.competitions.list_all():
            print(competition.id, competition.name, competition.max_levels)
    except AuthenticationError as exc:
        client = None
        print("Login failed:", exc)
else:
    print("API check skipped.")

API check skipped.


## Speech Transcriber

Whisper is local. It loads only when a speech live run starts.


In [5]:
from polimillionaire.transcribe import get_faster_whisper, get_whisper, make_transcriber, unload_whisper

speech_transcribe = make_transcriber(
    model_id=SPEECH_WHISPER_MODEL_ID,
    device=SPEECH_WHISPER_DEVICE,
    dtype=SPEECH_WHISPER_DTYPE,
    backend=SPEECH_WHISPER_BACKEND,
    fallback_model_id=SPEECH_WHISPER_FALLBACK_MODEL_ID,
    fallback_device=SPEECH_WHISPER_FALLBACK_DEVICE,
    fallback_dtype=SPEECH_WHISPER_FALLBACK_DTYPE,
    fallback_backend=SPEECH_WHISPER_FALLBACK_BACKEND,
    fallback_faster_compute_type=SPEECH_WHISPER_FALLBACK_COMPUTE_TYPE,
)

if RUN_LIVE_GAME:
    print("Loading Whisper before the speech timer starts...")
    if SPEECH_WHISPER_BACKEND == "faster_whisper":
        get_faster_whisper(
            model_id=SPEECH_WHISPER_MODEL_ID,
            device=SPEECH_WHISPER_DEVICE,
            compute_type=SPEECH_WHISPER_FALLBACK_COMPUTE_TYPE,
        )
    else:
        get_whisper(
            model_id=SPEECH_WHISPER_MODEL_ID,
            device=SPEECH_WHISPER_DEVICE,
            dtype=SPEECH_WHISPER_DTYPE,
        )
    if SPEECH_WHISPER_FALLBACK_MODEL_ID:
        if SPEECH_WHISPER_FALLBACK_BACKEND == "faster_whisper":
            get_faster_whisper(
                model_id=SPEECH_WHISPER_FALLBACK_MODEL_ID,
                device=SPEECH_WHISPER_FALLBACK_DEVICE,
                compute_type=SPEECH_WHISPER_FALLBACK_COMPUTE_TYPE,
            )
        else:
            get_whisper(
                model_id=SPEECH_WHISPER_FALLBACK_MODEL_ID,
                device=SPEECH_WHISPER_FALLBACK_DEVICE,
                dtype=SPEECH_WHISPER_FALLBACK_DTYPE,
            )
    try:
        from polimillionaire.transcribe import get_vad
        get_vad()
        print("Silero VAD ready.")
    except Exception as exc:
        print("Silero VAD unavailable; raw Whisper fallback will be used:", exc)
    print("Speech transcriber ready.")
else:
    print("Speech live runs disabled. Whisper not loaded.")


Loading Whisper before the speech timer starts...
Loading openai/whisper-base.en on cpu ...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

Whisper loaded on cpu (checkpoint max_length=448)
Loading faster-whisper distil-large-v3 on cpu (int8) ...
faster-whisper loaded on cpu
Silero VAD unavailable; raw Whisper fallback will be used: No module named 'silero_vad'
Speech transcriber ready.


## Speech Replay Check

Quick ASR check on saved speech WAVs before touching the live API.


In [6]:
import re
from collections import defaultdict


def collect_speech_audio_sets(audio_root):
    groups = defaultdict(dict)
    if not audio_root.exists():
        return []
    pattern = re.compile(r"level_(\d+)_question_(\d+)_(question|option_[A-D])\.wav$")
    for path in audio_root.rglob("*.wav"):
        match = pattern.match(path.name)
        if not match:
            continue
        level, question_id, label = match.groups()
        groups[(path.parent, int(level), int(question_id))][label] = path
    complete = []
    for (folder, level, question_id), files in groups.items():
        option_paths = [files.get(f"option_{letter}") for letter in "ABCD"]
        if files.get("question") and all(option_paths):
            complete.append({"folder": folder, "level": level, "question_id": question_id, "question": files["question"], "options": option_paths})
    return sorted(complete, key=lambda item: item["question"].stat().st_mtime, reverse=True)


def run_speech_replay_check():
    if not RUN_SPEECH_REPLAY_BENCHMARK:
        print("Speech replay check skipped.")
        return True
    audio_sets = collect_speech_audio_sets(SPEECH_AUDIO_ROOT)
    if not audio_sets:
        print("No saved speech WAVs found yet; replay check skipped.")
        return True
    rows = []
    for item in audio_sets[:SPEECH_REPLAY_MAX_AUDIO_SETS]:
        started = time.monotonic()
        question_text = speech_transcribe(item["question"].read_bytes())
        option_texts = [speech_transcribe(path.read_bytes()) for path in item["options"]]
        seconds = time.monotonic() - started
        empty_count = int(not question_text) + sum(1 for text in option_texts if not text)
        placeholder_count = sum(1 for text in option_texts if text.strip().lower() in {"option a", "option b", "option c", "option d"})
        ok = empty_count == 0 and placeholder_count == 0 and seconds <= SPEECH_REPLAY_MAX_SECONDS_PER_SET
        rows.append({"ok": ok, "seconds": seconds, "empty_count": empty_count, "placeholder_count": placeholder_count})
        print("Replay", item["folder"].name, "level", item["level"], "qid", item["question_id"], "ok:", ok, "seconds:", round(seconds, 2))
        print("  Q:", question_text[:180])
        print("  Options:", " | ".join(text[:60] for text in option_texts))
    passed = all(row["ok"] for row in rows)
    print("Speech replay gate:", "pass" if passed else "fail", rows)
    return passed


SPEECH_REPLAY_OK = run_speech_replay_check()


[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Replay category_best_5_news_gemma_e2b_two_agent_competition_5_20260601_135317 level 0 qid 11773 ok: True seconds: 2.94
  Q: According to the article published on 2026-0516, which two leaders traveled together to an international summit for the first time?
  Options: President Maya Sandhu and President Emmanuel Macron. | President Maya Saindou and President Vladimir Putin | President Maya Sandu and President Nikasore Dan. | President Maya Sandu and President Volodymyr Zelensky.
Replay category_best_4_philosophy_psychology_gemma_e2b_routed_rag_competition_4_20260601_135019 level 0 qid 9070 ok: True seconds: 7.08
  Q: How does physical conservatism relate to classical liberalism?
  Options: fiscal conservatism is entirely separate from classical libe | Fiscal conservatism opposes the principles of classical libe | Fiscal conservatism hurting its hollow hooded bed. That's no | fiscal conservatism rejects the economic principles of class
Speech replay gate: pass [{'ok': True, 'seconds': 2.9

## Benchmark Questions

In [7]:
import re

from polimillionaire.runner import RunLogger, SpeechGameRunner, load_jsonl, summarize_attempts
from polimillionaire.strategies import CalculatorStrategy, CouncilStrategy, GemmaLLM, GemmaLLMConfig, GemmaStrategy, LocalLLMVariant, QwenLLM, QwenLLMConfig, QwenStrategy, LangChainAgenticStrategy, HeuristicStrategy, RAGStrategy, RAGConfig, RoutedRAGCouncilStrategy, RoutedStrategy, route_question, unload_rag_runtime, unload_strategy
from polimillionaire.types import AnswerOption, Question

warmup_question = Question(1, "What is 2 + 2?", [AnswerOption(0, "3"), AnswerOption(1, "4"), AnswerOption(2, "5"), AnswerOption(3, "22")])
rag_warmup_question = Question(4, "In which year was A Haunted House 2 released?", [AnswerOption(0, "2012"), AnswerOption(1, "2015"), AnswerOption(2, "2014"), AnswerOption(3, "2013")])
TUTORIAL_SET = [
    (warmup_question, 1),
    (rag_warmup_question, 2),
    (Question(2, "Which song was NOT written by Bob Dylan?", [AnswerOption(0, "Like a Rolling Stone"), AnswerOption(1, "Blowin' in the Wind"), AnswerOption(2, "The Times They Are A-Changin'"), AnswerOption(3, "Hound Dog")]), 3),
    (Question(3, "What was Whitney Houston's debut album?", [AnswerOption(0, "Whitney Houston"), AnswerOption(1, "Just Whitney"), AnswerOption(2, "I'm Your Baby Tonight"), AnswerOption(3, "Whitney")]), 0),
    (Question(5, "Compute \\dbinom{85}{82}.", [AnswerOption(0, "252"), AnswerOption(1, "101170"), AnswerOption(2, "98770"), AnswerOption(3, "4680")]), 2),
    (Question(6, "Which genetics principle says alleles for one trait separate independently from alleles for another trait?", [AnswerOption(0, "Law of independent assortment"), AnswerOption(1, "Law of dominance"), AnswerOption(2, "Genetic drift"), AnswerOption(3, "Mitochondrial inheritance")]), 0),
    (Question(7, "Which of the following is a key characteristic of hard bop?", [AnswerOption(0, "Incorporation of blues and gospel elements"), AnswerOption(1, "Use of electronic synthesizers"), AnswerOption(2, "No structured form"), AnswerOption(3, "Strict adherence to 4/4 time")]), 0),
    (Question(8, "For a Japan-set racing game, which research step best supports authentic representation?", [AnswerOption(0, "Only copying older racing games"), AnswerOption(1, "Working with local cultural consultants and doing field research in Japan"), AnswerOption(2, "Using random online comments"), AnswerOption(3, "Avoiding real Japanese locations")]), 1),
    (Question(9, "In bystander-effect research, how can strong group cohesiveness change helping behavior?", [AnswerOption(0, "It can make people more likely to help members of the group"), AnswerOption(1, "It always removes personal responsibility"), AnswerOption(2, "It makes intervention impossible"), AnswerOption(3, "It has no relationship to helping")]), 0),
    (Question(10, "What is the usual link between the Battle of Marathon and the Older Parthenon?", [AnswerOption(0, "The temple project is linked to Athens after its victory at Marathon"), AnswerOption(1, "The battle was fought inside the Parthenon"), AnswerOption(2, "The Parthenon was built by Sparta after Marathon"), AnswerOption(3, "The Older Parthenon was a Roman project")]), 0),
    (Question(11, "Why was Hitchcock involved with the British Ministry of Information documentary about liberated concentration camps?", [AnswerOption(0, "To make a comedy film"), AnswerOption(1, "To help present visual evidence of Nazi atrocities clearly"), AnswerOption(2, "To advertise a commercial movie"), AnswerOption(3, "To hide the camps from the public")]), 1),
]
CATEGORY_BENCHMARKS = {
    0: [
        TUTORIAL_SET[2],
        TUTORIAL_SET[3],
        TUTORIAL_SET[6],
        TUTORIAL_SET[10],
    ],
    1: [
        TUTORIAL_SET[9],
        (Question(15, "Which event is usually associated with the end of the Roman Republic and the rise of the Roman Empire?", [AnswerOption(0, "The accession of Augustus as first emperor"), AnswerOption(1, "The founding of Rome"), AnswerOption(2, "The fall of Constantinople"), AnswerOption(3, "The Punic Wars")]), 0),
    ],
    2: [
        TUTORIAL_SET[5],
        (Question(12, "An organ pipe produces a note with wavelength 2.67 m. If the speed of sound is 343 m/s, what is the frequency?", [AnswerOption(0, "85.7 Hz"), AnswerOption(1, "128 Hz"), AnswerOption(2, "256 Hz"), AnswerOption(3, "343 Hz")]), 1),
        (Question(16, "Materials such as carbon and nitrogen move repeatedly through living things and the environment. What is this process called?", [AnswerOption(0, "A biogeochemical cycle"), AnswerOption(1, "A chemical explosion"), AnswerOption(2, "A nuclear chain reaction"), AnswerOption(3, "A food label")]), 0),
    ],
    3: [
        TUTORIAL_SET[4],
        (Question(13, "How many homomorphisms are there of Z into Z_2?", [AnswerOption(0, "infinitely many"), AnswerOption(1, "0"), AnswerOption(2, "1"), AnswerOption(3, "2")]), 3),
        (Question(14, "A producer compares acne scores on the same people before and after a new cream. Which test is appropriate?", [AnswerOption(0, "A two-sample t-test"), AnswerOption(1, "A matched pairs t-test"), AnswerOption(2, "A chi-square test"), AnswerOption(3, "A two-proportion z-test")]), 1),
        (Question(17, "What is 1^2 + 2^2 + 3^2 + 4^2?", [AnswerOption(0, "16"), AnswerOption(1, "20"), AnswerOption(2, "30"), AnswerOption(3, "40")]), 2),
    ],
    4: [
        TUTORIAL_SET[8],
        (Question(18, "What is the main difference between realism and anti-realism in philosophy?", [AnswerOption(0, "Realism says reality exists independently of us; anti-realism ties truth to minds or theories"), AnswerOption(1, "Both deny any external world"), AnswerOption(2, "Realism is only about painting"), AnswerOption(3, "Anti-realism says science is always impossible")]), 0),
        (Question(19, "Which idea is central to Stoic ethics?", [AnswerOption(0, "Virtue is the only true good"), AnswerOption(1, "Pleasure is the only good"), AnswerOption(2, "Wealth is the only good"), AnswerOption(3, "Fame is the only good")]), 0),
    ],
    5: [
        (Question(20, "According to this report summary, a broadcast deal stalled because regulators had not approved the rights package. What reason does the summary give?", [AnswerOption(0, "Regulators had not approved the rights package"), AnswerOption(1, "The sport was cancelled"), AnswerOption(2, "The teams refused to play"), AnswerOption(3, "There were too many stadiums")]), 0),
        (Question(21, "According to this news summary, a housing shortage mainly affected lower and middle price ranges. Which range was affected?", [AnswerOption(0, "Only luxury apartments"), AnswerOption(1, "Lower and middle price ranges"), AnswerOption(2, "Only vacation homes"), AnswerOption(3, "No price range")]), 1),
    ],
}
CATEGORY_PROBES = CATEGORY_BENCHMARKS
BENCHMARK_SET = [(category, question, gold_id) for category, items in CATEGORY_BENCHMARKS.items() for question, gold_id in items]


## Model Cleanup

In [8]:
def clear_model_memory(strategy=None, release_rag=False):
    try:
        unload_strategy(strategy)
        if release_rag:
            unload_rag_runtime()
        gc.collect()
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except Exception as exc:
        print("Cleanup warning:", exc)


## Architecture Builders

In [9]:
def mixed_council(quantized=False):
    if quantized and importlib.util.find_spec("bitsandbytes") is None:
        raise RuntimeError("For the 4-bit council, run `%pip install -U bitsandbytes`, restart the kernel, and rerun setup.")
    gemma = GemmaLLM(GemmaLLMConfig(model_id="google/gemma-4-E2B-it", quantize_4bit=quantized, max_new_tokens=32, do_sample=True, seed=42, generation_max_time_seconds=6.0, timeout_seconds=120.0))
    qwen = QwenLLM(QwenLLMConfig(model_id="Qwen/Qwen3.5-2B", quantize_4bit=quantized, max_new_tokens=32, temperature=0.7, top_p=0.8, top_k=20, do_sample=True, seed=43, enable_thinking=False, generation_max_time_seconds=6.0, timeout_seconds=120.0))
    return CouncilStrategy(candidate_llms=[gemma, qwen], judge_llm=gemma, judge_scope="candidate_only" if quantized else "any_option", rejected_judge_fallback="primary_candidate" if quantized else "confidence_weighted", base_seed=200, candidate_max_new_tokens=32, judge_max_new_tokens=8, max_time_per_call=6.0)


def mixed_quantized_routed_rag():
    if importlib.util.find_spec("bitsandbytes") is None:
        raise RuntimeError("For 4-bit mixed routed RAG, run `%pip install -U bitsandbytes`, restart the kernel, and rerun setup.")
    gemma = GemmaLLM(GemmaLLMConfig(model_id="google/gemma-4-E2B-it", quantize_4bit=True, max_new_tokens=48, temperature=0.0, do_sample=False, seed=42, generation_max_time_seconds=12.0, timeout_seconds=120.0))
    qwen = QwenLLM(QwenLLMConfig(model_id="Qwen/Qwen3.5-2B", quantize_4bit=True, max_new_tokens=48, temperature=0.7, top_p=0.8, top_k=20, do_sample=True, seed=43, enable_thinking=False, generation_max_time_seconds=10.0, timeout_seconds=120.0))
    gemma_direct = LocalLLMVariant(gemma, "gemma-e2b-direct", do_sample=False, temperature=0.0, seed=42, max_time=8.0)
    gemma_candidate = LocalLLMVariant(gemma, "gemma-e2b-candidate", do_sample=True, temperature=0.55, top_p=0.9, seed=201, max_time=10.0)
    direct = CalculatorStrategy(GemmaStrategy(llm=gemma_direct))
    council = CouncilStrategy(candidate_llms=[gemma_candidate, qwen], judge_llm=gemma_direct, judge_scope="candidate_only", rejected_judge_fallback="primary_candidate", base_seed=200, candidate_max_new_tokens=48, judge_max_new_tokens=8, max_time_per_call=10.0)
    rag = RAGStrategy(llm=gemma_direct, retrieval_config=RAGConfig(num_extra_queries=0, answer_max_new_tokens=80), fallback_strategy=council)
    return RoutedStrategy(direct_strategy=direct, rag_strategy=rag, fallback_strategy=council, low_confidence_strategy=council, rag_min_confidence=0.65)


def gemma_e2b_rag_council_e4b_judge():
    if importlib.util.find_spec("bitsandbytes") is None:
        raise RuntimeError("For the Gemma council, run `%pip install -U bitsandbytes`, restart the kernel, and rerun setup.")
    e2b = GemmaLLM(GemmaLLMConfig(model_id="google/gemma-4-E2B-it", quantize_4bit=True, max_new_tokens=80, temperature=0.7, do_sample=True, generation_max_time_seconds=12.0, timeout_seconds=120.0))
    e4b_judge = GemmaLLM(GemmaLLMConfig(model_id="google/gemma-4-E4B-it", quantize_4bit=True, max_new_tokens=16, temperature=0.0, do_sample=False, seed=900, generation_max_time_seconds=10.0, timeout_seconds=120.0))
    candidates = [
        LocalLLMVariant(e2b, "gemma-e2b-agent-a", do_sample=True, temperature=0.35, top_p=0.9, seed=501, max_time=10.0),
        LocalLLMVariant(e2b, "gemma-e2b-agent-b", do_sample=True, temperature=0.65, top_p=0.9, seed=502, max_time=10.0),
        LocalLLMVariant(e2b, "gemma-e2b-agent-c", do_sample=True, temperature=0.9, top_p=0.95, seed=503, max_time=10.0),
    ]
    direct = CalculatorStrategy(GemmaStrategy(llm=LocalLLMVariant(e2b, "gemma-e2b-direct", do_sample=False, temperature=0.0, seed=42, max_time=8.0)))
    return RoutedRAGCouncilStrategy(
        candidate_llms=candidates,
        judge_llm=e4b_judge,
        direct_strategy=direct,
        retrieval_config=RAGConfig(num_extra_queries=0, max_ddg_results=3, timeout_ddg=4, fetch_timeout=3.0, total_fetch_budget=3.0, bm25_top_k=4, dense_top_k=4, final_top_k=2, answer_max_new_tokens=48),
        judge_scope="candidate_only",
        base_seed=600,
        candidate_max_new_tokens=48,
        judge_max_new_tokens=8,
        max_time_per_call=8.0,
        always_judge=False,
        unload_candidates_before_judge=True,
    )



def gemma_e2b_quantized_council_e2b_normal_judge():
    if importlib.util.find_spec("bitsandbytes") is None:
        raise RuntimeError("For this council, run `%pip install -U bitsandbytes`, restart the kernel, and rerun setup.")
    e2b = GemmaLLM(GemmaLLMConfig(model_id="google/gemma-4-E2B-it", quantize_4bit=True, max_new_tokens=48, temperature=0.7, do_sample=True, generation_max_time_seconds=8.0, timeout_seconds=120.0))
    candidates = [
        LocalLLMVariant(e2b, "gemma-e2b-4bit-evidence-checker", do_sample=True, temperature=0.35, top_p=0.9, seed=701, max_time=8.0),
        LocalLLMVariant(e2b, "gemma-e2b-4bit-option-eliminator", do_sample=True, temperature=0.75, top_p=0.95, seed=702, max_time=8.0),
    ]
    judge = LocalLLMVariant(e2b, "gemma-e2b-4bit-judge", do_sample=False, temperature=0.0, seed=900, max_time=8.0)
    direct = CalculatorStrategy(GemmaStrategy(llm=LocalLLMVariant(e2b, "gemma-e2b-4bit-direct", do_sample=False, temperature=0.0, seed=42, max_time=8.0)))
    return RoutedRAGCouncilStrategy(
        candidate_llms=candidates,
        judge_llm=judge,
        direct_strategy=direct,
        retrieval_config=RAGConfig(num_extra_queries=0, max_ddg_results=3, timeout_ddg=4, fetch_timeout=3.0, total_fetch_budget=3.0, bm25_top_k=4, dense_top_k=4, final_top_k=2, answer_max_new_tokens=48),
        judge_scope="candidate_only",
        base_seed=700,
        candidate_styles=["evidence_checker", "option_eliminator"],
        candidate_max_new_tokens=48,
        judge_max_new_tokens=8,
        max_time_per_call=8.0,
        always_judge=False,
        unload_candidates_before_judge=False,
        unload_judge_before_candidates=False,
    )


def qwen_quantized_rag_council():
    if importlib.util.find_spec("bitsandbytes") is None:
        raise RuntimeError("For this Qwen council, run `%pip install -U bitsandbytes`, restart the kernel, and rerun setup.")
    qwen = QwenLLM(QwenLLMConfig(model_id="Qwen/Qwen3.5-2B", quantize_4bit=True, max_new_tokens=48, temperature=0.7, top_p=0.9, top_k=20, do_sample=True, seed=42, enable_thinking=False, generation_max_time_seconds=8.0, timeout_seconds=120.0))
    candidates = [
        LocalLLMVariant(qwen, "qwen-3.5-2b-4bit-evidence-checker", do_sample=True, temperature=0.25, top_p=0.9, top_k=20, seed=801, max_time=8.0),
        LocalLLMVariant(qwen, "qwen-3.5-2b-4bit-option-eliminator", do_sample=True, temperature=0.75, top_p=0.95, top_k=20, seed=802, max_time=8.0),
    ]
    judge = LocalLLMVariant(qwen, "qwen-3.5-2b-4bit-judge", do_sample=False, seed=950, max_time=8.0)
    direct_llm = LocalLLMVariant(qwen, "qwen-3.5-2b-4bit-direct", do_sample=False, seed=42, max_time=8.0)
    direct = CalculatorStrategy(QwenStrategy(llm=direct_llm, model_config=QwenLLMConfig(enable_thinking=False)))
    return RoutedRAGCouncilStrategy(
        candidate_llms=candidates,
        judge_llm=judge,
        direct_strategy=direct,
        retrieval_config=RAGConfig(num_extra_queries=0, max_ddg_results=3, timeout_ddg=4, fetch_timeout=3.0, total_fetch_budget=3.0, bm25_top_k=4, dense_top_k=4, final_top_k=2, answer_max_new_tokens=48),
        judge_scope="candidate_only",
        base_seed=800,
        candidate_styles=["evidence_checker", "option_eliminator"],
        candidate_max_new_tokens=48,
        judge_max_new_tokens=8,
        max_time_per_call=8.0,
        always_judge=False,
        unload_candidates_before_judge=False,
        unload_judge_before_candidates=False,
    )

def gemma_quantized_routed_rag(model_id="google/gemma-4-E2B-it", label="gemma-e2b"):
    if importlib.util.find_spec("bitsandbytes") is None:
        raise RuntimeError("For Gemma routed RAG, run `%pip install -U bitsandbytes`, restart the kernel, and rerun setup.")
    gemma = GemmaLLM(GemmaLLMConfig(model_id=model_id, quantize_4bit=True, max_new_tokens=48, temperature=0.0, do_sample=False, seed=42, generation_max_time_seconds=10.0, timeout_seconds=120.0))
    direct_llm = LocalLLMVariant(gemma, f"{label}-rag-direct", do_sample=False, temperature=0.0, seed=42, max_time=8.0)
    direct = CalculatorStrategy(GemmaStrategy(llm=direct_llm))
    rag = RAGStrategy(
        llm=direct_llm,
        retrieval_config=RAGConfig(num_extra_queries=0, max_ddg_results=3, timeout_ddg=4, fetch_timeout=3.0, total_fetch_budget=3.0, bm25_top_k=4, dense_top_k=4, final_top_k=2, answer_max_new_tokens=64),
        fallback_strategy=direct,
    )
    return RoutedStrategy(direct_strategy=direct, rag_strategy=rag, fallback_strategy=direct, low_confidence_strategy=direct, rag_min_confidence=0.55)


def qwen_quantized_tool_rag():
    if importlib.util.find_spec("bitsandbytes") is None:
        raise RuntimeError("For Qwen routed RAG, run `%pip install -U bitsandbytes`, restart the kernel, and rerun setup.")
    qwen = QwenLLM(QwenLLMConfig(model_id="Qwen/Qwen3.5-2B", quantize_4bit=True, max_new_tokens=48, temperature=0.0, do_sample=False, seed=42, enable_thinking=False, generation_max_time_seconds=10.0, timeout_seconds=120.0))
    direct_llm = LocalLLMVariant(qwen, "qwen-3.5-2b-rag-direct", do_sample=False, seed=42, max_time=8.0)
    direct = CalculatorStrategy(QwenStrategy(llm=direct_llm, model_config=QwenLLMConfig(enable_thinking=False)))
    rag = RAGStrategy(
        llm=direct_llm,
        retrieval_config=RAGConfig(num_extra_queries=0, max_ddg_results=3, timeout_ddg=4, fetch_timeout=3.0, total_fetch_budget=3.0, bm25_top_k=4, dense_top_k=4, final_top_k=2, answer_max_new_tokens=64),
        fallback_strategy=direct,
    )
    return RoutedStrategy(direct_strategy=direct, rag_strategy=rag, fallback_strategy=direct, low_confidence_strategy=direct, rag_min_confidence=0.55)


class LazyFrankenFallback:
    name = "lazy_franken_fallback"

    def __init__(self, owner, key):
        self.owner = owner
        self.key = key

    def answer(self, question):
        return self.owner._get(self.key).answer(question)


class DataRoutedFrankenStrategy:
    name = "data_routed_franken"

    def __init__(self):
        self._loaded = {}
        self._models = {}

    def loaded_strategies(self):
        return list(self._loaded.values())

    def unload(self):
        for strategy in list(self._loaded.values()):
            unload_strategy(strategy)
        for model in list(self._models.values()):
            unload_strategy(model)
        unload_rag_runtime()
        self._loaded.clear()
        self._models.clear()

    def _gemma_llm(self):
        if "gemma" not in self._models:
            self._models["gemma"] = GemmaLLM(GemmaLLMConfig(
                model_id="google/gemma-4-E2B-it",
                quantize_4bit=True,
                max_new_tokens=48,
                temperature=0.0,
                do_sample=False,
                seed=42,
                generation_max_time_seconds=10.0,
                timeout_seconds=120.0,
            ))
        return self._models["gemma"]

    def _qwen_llm(self):
        if "qwen" not in self._models:
            self._models["qwen"] = QwenLLM(QwenLLMConfig(
                model_id="Qwen/Qwen3.5-2B",
                quantize_4bit=True,
                max_new_tokens=48,
                temperature=0.7,
                top_p=0.9,
                top_k=20,
                do_sample=True,
                seed=42,
                enable_thinking=False,
                generation_max_time_seconds=8.0,
                timeout_seconds=120.0,
            ))
        return self._models["qwen"]

    def _build_gemma_rag(self):
        gemma = self._gemma_llm()
        direct_llm = LocalLLMVariant(gemma, "franken-gemma-rag-direct", do_sample=False, temperature=0.0, seed=42, max_time=8.0)
        direct = CalculatorStrategy(GemmaStrategy(llm=direct_llm))
        rag = RAGStrategy(
            llm=direct_llm,
            retrieval_config=RAGConfig(num_extra_queries=0, max_ddg_results=3, timeout_ddg=4, fetch_timeout=3.0, total_fetch_budget=3.0, bm25_top_k=4, dense_top_k=4, final_top_k=2, answer_max_new_tokens=64),
            fallback_strategy=direct,
        )
        return RoutedStrategy(direct_strategy=direct, rag_strategy=rag, fallback_strategy=direct, low_confidence_strategy=direct, rag_min_confidence=0.55)

    def _build_gemma_council(self):
        gemma = self._gemma_llm()
        candidates = [
            LocalLLMVariant(gemma, "franken-gemma-evidence-checker", do_sample=True, temperature=0.35, top_p=0.9, seed=701, max_time=8.0),
            LocalLLMVariant(gemma, "franken-gemma-option-eliminator", do_sample=True, temperature=0.75, top_p=0.95, seed=702, max_time=8.0),
        ]
        judge = LocalLLMVariant(gemma, "franken-gemma-judge", do_sample=False, temperature=0.0, seed=900, max_time=8.0)
        direct = CalculatorStrategy(GemmaStrategy(llm=LocalLLMVariant(gemma, "franken-gemma-direct", do_sample=False, temperature=0.0, seed=42, max_time=8.0)))
        return RoutedRAGCouncilStrategy(
            candidate_llms=candidates,
            judge_llm=judge,
            direct_strategy=direct,
            retrieval_config=RAGConfig(num_extra_queries=0, max_ddg_results=3, timeout_ddg=4, fetch_timeout=3.0, total_fetch_budget=3.0, bm25_top_k=4, dense_top_k=4, final_top_k=2, answer_max_new_tokens=48),
            judge_scope="candidate_only",
            base_seed=700,
            candidate_styles=["evidence_checker", "option_eliminator"],
            candidate_max_new_tokens=48,
            judge_max_new_tokens=8,
            max_time_per_call=8.0,
            always_judge=False,
            unload_candidates_before_judge=False,
            unload_judge_before_candidates=False,
        )

    def _build_qwen_council(self):
        qwen = self._qwen_llm()
        candidates = [
            LocalLLMVariant(qwen, "franken-qwen-evidence-checker", do_sample=True, temperature=0.25, top_p=0.9, top_k=20, seed=801, max_time=8.0),
            LocalLLMVariant(qwen, "franken-qwen-option-eliminator", do_sample=True, temperature=0.75, top_p=0.95, top_k=20, seed=802, max_time=8.0),
        ]
        judge = LocalLLMVariant(qwen, "franken-qwen-judge", do_sample=False, seed=950, max_time=8.0)
        direct_llm = LocalLLMVariant(qwen, "franken-qwen-direct", do_sample=False, seed=42, max_time=8.0)
        direct = CalculatorStrategy(QwenStrategy(llm=direct_llm, model_config=QwenLLMConfig(enable_thinking=False)))
        return RoutedRAGCouncilStrategy(
            candidate_llms=candidates,
            judge_llm=judge,
            direct_strategy=direct,
            retrieval_config=RAGConfig(num_extra_queries=0, max_ddg_results=3, timeout_ddg=4, fetch_timeout=3.0, total_fetch_budget=3.0, bm25_top_k=4, dense_top_k=4, final_top_k=2, answer_max_new_tokens=48),
            judge_scope="candidate_only",
            base_seed=800,
            candidate_styles=["evidence_checker", "option_eliminator"],
            candidate_max_new_tokens=48,
            judge_max_new_tokens=8,
            max_time_per_call=8.0,
            always_judge=False,
            unload_candidates_before_judge=False,
            unload_judge_before_candidates=False,
        )

    def _get(self, key):
        if key not in self._loaded:
            if key == "gemma_council":
                self._loaded[key] = self._build_gemma_council()
            elif key == "qwen_council":
                self._loaded[key] = self._build_qwen_council()
            elif key == "gemma_rag":
                self._loaded[key] = self._build_gemma_rag()
            elif key == "tools":
                self._loaded[key] = CalculatorStrategy(fallback_strategy=LazyFrankenFallback(self, "gemma_council"))
            else:
                raise KeyError(key)
        return self._loaded[key]

    def _route(self, question):
        text = " ".join([question.text, *(option.text for option in question.options)]).lower()
        decision = route_question(question)
        if decision.reason == "math" or any(term in text for term in ["frequency", "wavelength", "homomorphism", "significance test", "matched pairs", "binom", "dbinom", "linear transformation", "confidence interval", "interquartile range", "decreased linearly"]):
            return "tools", "tool_math_or_formula"
        if re.search(r"\b20\d{2}[-/]\d{1,2}[-/]\d{1,2}\b", text) or any(term in text for term in ["according to the report", "news article", "broadcasting deal", "reported that", "coach stated"]):
            return "gemma_rag", "news_or_report_gemma_rag"
        if any(term in text for term in ["philosophy", "psychology", "bystander", "realism", "anti-realism", "denial", "santayana", "aristotle", "kant", "nietzsche", "plato", "blindsight", "relationship satisfaction"]):
            return "qwen_council", "philosophy_or_psych_qwen"
        if any(term in text for term in ["allele", "genetic", "biology", "cloud", "precipitation", "organ pipe", "wavelength", "molecule", "embryo", "frequency", "speed of sound", "planet", "solar system", "reef", "ecosystem"]):
            return "gemma_council", "science_gemma_council"
        if any(term in text for term in ["film", "movie", "song", "album", "music", "actor", "actress", "director", "singer", "dylan", "hitchcock", "kubrick", "cameron"]):
            return "gemma_rag", "entertainment_gemma_rag"
        if any(term in text for term in ["roman", "ancient", "alexander", "egyptian", "empire", "historical", "history"]):
            return "gemma_council", "history_gemma_council"
        return "gemma_council", "default_gemma"

    def answer(self, question):
        key, reason = self._route(question)
        prediction = self._get(key).answer(question)
        prediction.metadata["franken_component"] = key
        prediction.metadata["franken_route"] = reason
        return prediction


    def preload_components(self):
        probes = [
            ("tools", warmup_question, 1),
            ("gemma_rag", Question(6, "Which genetics principle says alleles for one trait separate independently from alleles for another trait?", [AnswerOption(0, "Law of independent assortment"), AnswerOption(1, "Law of dominance"), AnswerOption(2, "Genetic drift"), AnswerOption(3, "Mitochondrial inheritance")]), 0),
            ("qwen_council", Question(9, "In bystander-effect research, how can strong group cohesiveness change helping behavior?", [AnswerOption(0, "It can make people more likely to help members of the group"), AnswerOption(1, "It always removes personal responsibility"), AnswerOption(2, "It makes intervention impossible"), AnswerOption(3, "It has no relationship to helping")]), 0),
            ("gemma_council", Question(11, "Why was Hitchcock involved with the British Ministry of Information documentary about liberated concentration camps?", [AnswerOption(0, "To make a comedy film"), AnswerOption(1, "To help present visual evidence of Nazi atrocities clearly"), AnswerOption(2, "To advertise a commercial movie"), AnswerOption(3, "To hide the camps from the public")]), 1),
        ]
        rows = []
        for component, question, gold_id in probes:
            started = time.monotonic()
            prediction = self.answer(question)
            seconds = time.monotonic() - started
            rows.append({
                "component": component,
                "seconds": round(seconds, 2),
                "correct": prediction.option_id == gold_id,
                "fallback": bool(prediction.metadata.get("fallback")),
                "predicted": prediction.option_id,
                "gold": gold_id,
            })
        return rows

## Strategy Factory

In [10]:
def has_rag_strategy(strategy):
    if isinstance(strategy, DataRoutedFrankenStrategy):
        return True
    if isinstance(strategy, (RAGStrategy, RoutedRAGCouncilStrategy)):
        return True
    if isinstance(strategy, RoutedStrategy):
        return has_rag_strategy(strategy.rag_strategy)
    return False


def strategy_devices(strategy):
    devices = []

    def collect(item):
        if item is None:
            return
        if isinstance(item, DataRoutedFrankenStrategy):
            for loaded in item.loaded_strategies():
                collect(loaded)
            return
        if isinstance(item, CouncilStrategy):
            for llm in item.candidate_llms + [item.judge_llm]:
                devices.append(getattr(llm, "device_summary", "unknown"))
            return
        if isinstance(item, RoutedRAGCouncilStrategy):
            collect(item.direct_strategy)
            for llm in item.candidate_llms + [item.judge_llm]:
                devices.append(getattr(llm, "device_summary", "unknown"))
            return
        if isinstance(item, RoutedStrategy):
            collect(item.direct_strategy)
            collect(item.rag_strategy)
            collect(item.fallback_strategy)
            collect(item.low_confidence_strategy)
            return
        if hasattr(item, "llm"):
            devices.append(getattr(item.llm, "device_summary", "unknown"))

    collect(strategy)
    return list(dict.fromkeys(devices))


QUANTIZED_KINDS = {"gemma_quantized", "mixed_quantized", "mixed_quantized_rag", "gemma_e2b_rag_council_e4b_judge", "gemma_e2b_quantized_council_e2b_normal_judge", "qwen_quantized_rag_council", "franken_data_route", "rag", "gemma_tool_rag_quant", "gemma_e4b_tool_rag_quant", "qwen_tool_rag_quant", "gemma_tool_council_quant", "qwen_tool_council_quant"}


def make_strategy(item):
    if item["kind"] == "tool_heuristic":
        fallback = HeuristicStrategy()
        return RoutedStrategy(direct_strategy=CalculatorStrategy(fallback), fallback_strategy=fallback)
    if item["kind"] == "gemma_tool_rag_quant":
        return gemma_quantized_routed_rag(model_id="google/gemma-4-E2B-it", label="gemma-e2b")
    if item["kind"] == "gemma_e4b_tool_rag_quant":
        return gemma_quantized_routed_rag(model_id="google/gemma-4-E4B-it", label="gemma-e4b")
    if item["kind"] == "qwen_tool_rag_quant":
        return qwen_quantized_tool_rag()
    if item["kind"] == "gemma_tool_council_quant":
        return gemma_e2b_quantized_council_e2b_normal_judge()
    if item["kind"] == "qwen_tool_council_quant":
        return qwen_quantized_rag_council()
    if item["kind"] == "franken_data_route":
        return DataRoutedFrankenStrategy()
    if item["kind"] == "qwen_quantized_rag_council":
        return qwen_quantized_rag_council()
    if item["kind"] == "gemma_e2b_quantized_council_e2b_normal_judge":
        return gemma_e2b_quantized_council_e2b_normal_judge()
    if item["kind"] == "gemma_e2b_rag_council_e4b_judge":
        return gemma_e2b_rag_council_e4b_judge()
    if item["kind"] == "gemma_quantized":
        if importlib.util.find_spec("bitsandbytes") is None:
            raise RuntimeError("For 4-bit Gemma, run `%pip install -U bitsandbytes`, restart the kernel, and rerun setup.")
        return GemmaStrategy(model_config=GemmaLLMConfig(model_id=item["model_id"], quantize_4bit=True, max_new_tokens=8, temperature=0.0, do_sample=False, num_beams=1, seed=42, generation_max_time_seconds=18.0, timeout_seconds=120.0))
    if item["kind"] == "mixed_quantized":
        return mixed_council(quantized=True)
    if item["kind"] == "mixed_quantized_rag":
        return mixed_quantized_routed_rag()
    if item["kind"] == "langchain_agentic":
        if importlib.util.find_spec("bitsandbytes") is None:
            raise RuntimeError("For 4-bit LangChain Agent, run `%pip install -U bitsandbytes`.")
        
        agent_config = GemmaLLMConfig(
            model_id=item["model_id"], 
            quantize_4bit=True, 
            max_new_tokens=128,
            temperature=0.0, 
            do_sample=False, 
            generation_max_time_seconds=30.0, 
            timeout_seconds=120.0
        )
        base_llm = GemmaLLM(config=agent_config)
        return LangChainAgenticStrategy(raw_llm=base_llm, fallback_strategy=HeuristicStrategy())
    if item["kind"] == "mixed_council":
        return mixed_council()
    if item["kind"] == "gemma_council":
        llm = GemmaLLM(GemmaLLMConfig(model_id=item["model_id"], max_new_tokens=48, do_sample=True, seed=42, generation_max_time_seconds=4.5, timeout_seconds=120.0))
        return CouncilStrategy(llm=llm, num_votes=item.get("votes", 3), base_seed=100, candidate_max_new_tokens=48, judge_max_new_tokens=8, max_time_per_call=4.5)
    if item["kind"] == "qwen_thinking":
        return QwenStrategy(model_config=QwenLLMConfig(model_id=item["model_id"], max_new_tokens=128, temperature=1.0, top_p=0.95, top_k=20, do_sample=True, seed=42, enable_thinking=True, generation_max_time_seconds=18.0, timeout_seconds=120.0))
    if item["kind"] == "rag":
        if importlib.util.find_spec("bitsandbytes") is None:
            raise RuntimeError("For 4-bit RAG, run `%pip install -U bitsandbytes`, restart the kernel, and rerun setup.")
        base_llm = GemmaLLM(GemmaLLMConfig(
            model_id=item["model_id"],
            quantize_4bit=True,
            max_new_tokens=32,
            temperature=0.0,
            do_sample=False,
            generation_max_time_seconds=18.0,
            timeout_seconds=120.0,
        ))
        direct_strategy = GemmaStrategy(llm=base_llm)
        rag_strategy = RAGStrategy(
            llm=base_llm,
            retrieval_config=RAGConfig(num_extra_queries=0),  # disable expansion for speed
            fallback_strategy=HeuristicStrategy(),
        )
        return RoutedStrategy(direct_strategy=direct_strategy, rag_strategy=rag_strategy, fallback_strategy=direct_strategy)
    return GemmaStrategy(model_config=GemmaLLMConfig(model_id=item["model_id"], max_new_tokens=8, temperature=0.0, do_sample=False, num_beams=1, seed=42, generation_max_time_seconds=18.0, timeout_seconds=120.0))


## Benchmark Runner

In [11]:
def clean_name(label):
    return "_".join(label.lower().replace("+", " ").replace("(", " ").replace(")", " ").split())


benchmark_results = []


def benchmark(strategy, label, cases=None):
    cases = cases or BENCHMARK_SET
    rows = []
    category_rows = {}
    for item in cases:
        if len(item) == 3:
            category, question, gold_id = item
        else:
            category, question, gold_id = None, item[0], item[1]
        started = time.monotonic()
        prediction = strategy.answer(question)
        seconds = time.monotonic() - started
        correct = prediction.option_id == gold_id
        fallback = bool(prediction.metadata.get("fallback"))
        rows.append((category, correct, seconds, fallback))
        category_rows.setdefault(category, []).append((correct, seconds, fallback))
        gold = question.require_option(gold_id)
        print("\nC", category, "Q:", question.text)
        print("predicted:", prediction.option_id, prediction.answer_text)
        print("gold:", gold.id, gold.text, "correct:", correct, "seconds:", round(seconds, 2))
        print("decision:", prediction.metadata.get("decision_method"), "route:", prediction.metadata.get("route"), "franken:", prediction.metadata.get("franken_component"), prediction.metadata.get("franken_route"), "fallback:", prediction.metadata.get("fallback"))
        for index, vote in enumerate(prediction.metadata.get("votes") or [], start=1):
            print("vote", index, vote.get("model_name"), "style:", vote.get("style"), "->", vote.get("option_id"), "confidence:", vote.get("confidence"), "reason:", vote.get("reasoning"))
        if prediction.metadata.get("judge_raw_text") is not None:
            print("judge raw:", prediction.metadata.get("judge_raw_text"), "scope:", prediction.metadata.get("judge_scope"))
        for source in prediction.metadata.get("evidence_sources") or []:
            print("evidence:", source.get("title"), source.get("url"))
    accuracy = sum(correct for _, correct, _, _ in rows) / len(rows)
    max_seconds = max(seconds for _, _, seconds, _ in rows)
    all_correct = all(correct for _, correct, _, _ in rows)
    has_fallbacks = any(fallback for _, _, _, fallback in rows)
    wrong_questions = sum(1 for _, correct, _, _ in rows if not correct)
    by_category = {}
    for category, cat_rows in category_rows.items():
        by_category[category] = {
            "accuracy": sum(correct for correct, _, _ in cat_rows) / len(cat_rows),
            "avg_seconds": round(sum(seconds for _, seconds, _ in cat_rows) / len(cat_rows), 2),
            "max_seconds": round(max(seconds for _, seconds, _ in cat_rows), 2),
            "fallbacks": sum(fallback for _, _, fallback in cat_rows),
            "total": len(cat_rows),
        }
    summary = {"model": label, "accuracy": accuracy, "avg_seconds": round(sum(seconds for _, _, seconds, _ in rows) / len(rows), 2), "max_seconds": round(max_seconds, 2), "fallbacks": sum(fallback for _, _, _, fallback in rows), "all_correct": all_correct, "wrong_questions": wrong_questions, "by_category": by_category}
    benchmark_results.append(summary)
    print("Benchmark summary:", summary)
    print("Benchmark gate:", "pass" if all_correct and not has_fallbacks and max_seconds <= 20.0 else "fail")
    return all_correct and max_seconds <= 20.0 and not has_fallbacks


## Live Runs

The global-best run uses one architecture for every category. The category-best run switches architecture by competition id.


In [12]:
if RUN_LIVE_GAME and (not USERNAME or not PASSWORD):
    raise RuntimeError("Set credentials first.")
if RUN_LIVE_GAME and not SPEECH_REPLAY_OK and BLOCK_LIVE_ON_SPEECH_REPLAY_FAILURE:
    raise RuntimeError("Speech replay check failed. No live speech game started.")
if RUN_LIVE_GAME and client is None:
    client = MillionaireClient(API_URL)
    client.login(USERNAME, PASSWORD)


def benchmark_cases_for_competitions(competition_ids):
    selected = set(competition_ids)
    return [case for case in BENCHMARK_SET if case[0] in selected]


def make_live_plans():
    plans = []
    if RUN_BEST_SINGLE_ALL_CATEGORIES:
        item = dict(ARCHITECTURES[BEST_SINGLE_ARCHITECTURE_KEY])
        plans.append({
            "plan": "single_best_all_categories",
            "label": "Single best - " + item["label"],
            "log_label": "single_best_" + item["short_name"],
            "item": item,
            "competition_ids": list(COMPETITION_IDS),
            "benchmark_cases": BENCHMARK_SET,
        })
    if RUN_BEST_BY_CATEGORY:
        for competition_id in COMPETITION_IDS:
            key = BEST_BY_CATEGORY_KEYS[competition_id]
            item = dict(ARCHITECTURES[key])
            category_name = CATEGORY_NAMES[competition_id]
            plans.append({
                "plan": "best_by_category",
                "label": f"Category best {competition_id} {category_name} - {item['label']}",
                "log_label": f"category_best_{competition_id}_{category_name}_{item['short_name']}",
                "item": item,
                "competition_ids": [competition_id],
                "benchmark_cases": benchmark_cases_for_competitions([competition_id]),
            })
    return plans


run_results = []
live_plans = make_live_plans()
print("Selected speech live plans:", len(live_plans))

for plan in live_plans:
    item = plan["item"]
    strategy = None
    try:
        print("\nPlan:", plan["label"])
        print("Competitions:", plan["competition_ids"])
        strategy = make_strategy(item)
        started = time.monotonic()
        warmup = strategy.answer(warmup_question)
        load_and_warmup_seconds = time.monotonic() - started
        print("warmup option:", warmup.option_id, warmup.answer_text, "fallback:", warmup.metadata.get("fallback"), "route:", warmup.metadata.get("route"), "load + answer seconds:", round(load_and_warmup_seconds, 2))
        if warmup.metadata.get("fallback") or warmup.option_id != 1:
            raise RuntimeError("Warm-up failed. No live game started for this plan.")
        if item["kind"] in QUANTIZED_KINDS:
            print("initial devices:", strategy_devices(strategy))
        preload_rows = []
        if PRELOAD_SELECTED_STRATEGIES and hasattr(strategy, "preload_components"):
            started = time.monotonic()
            preload_rows = strategy.preload_components()
            preload_seconds = time.monotonic() - started
            print("preload seconds:", round(preload_seconds, 2))
            for row in preload_rows:
                print("preload", row)
            if any(row["fallback"] for row in preload_rows):
                raise RuntimeError("Component preload fallback. No live game started for this plan.")
        elif has_rag_strategy(strategy):
            started = time.monotonic()
            rag_warmup = strategy.answer(rag_warmup_question)
            rag_warmup_seconds = time.monotonic() - started
            print("rag warmup option:", rag_warmup.option_id, rag_warmup.answer_text, "fallback:", rag_warmup.metadata.get("fallback"), "route:", rag_warmup.metadata.get("route"), "seconds:", round(rag_warmup_seconds, 2))
            if rag_warmup.metadata.get("fallback") or rag_warmup.option_id != 2:
                raise RuntimeError("RAG warm-up failed. No live game started for this plan.")
        print("Benchmark timings below are warm timings; preload time is reported separately.")
        benchmark_ok = benchmark(strategy, plan["label"], cases=plan["benchmark_cases"]) if RUN_OFFLINE_BENCHMARK else True
        if item["kind"] in QUANTIZED_KINDS:
            devices = strategy_devices(strategy)
            print("loaded devices:", devices)
            loaded_devices = " ".join(device for device in devices if device != "not_loaded").lower()
            if any(token in loaded_devices for token in ("'cpu'", "'disk'", "meta")):
                raise RuntimeError("Quantized model was offloaded. No live game started for this plan.")
        if not benchmark_ok and RUN_LIVE_GAME and BLOCK_LIVE_ON_BENCHMARK_FAILURE:
            raise RuntimeError("Benchmark gate failed. No live game started.")
        if not benchmark_ok:
            print("Benchmark did not pass; continuing because benchmark gating is disabled.")
        result = {
            "plan": plan["plan"],
            "label": plan["label"],
            "architecture": item["label"],
            "competition_ids": plan["competition_ids"],
            "warmup_option": warmup.option_id,
            "benchmark_ok": benchmark_ok,
            "preload_rows": preload_rows,
        }
        if RUN_LIVE_GAME:
            live_runs = []
            for competition_id in plan["competition_ids"]:
                run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
                log_path = REPO_ROOT / "results" / "runs" / f"speech_{clean_name(plan['log_label'])}_competition_{competition_id}_{run_id}.jsonl"
                audio_dir = SPEECH_AUDIO_ROOT / f"{clean_name(plan['log_label'])}_competition_{competition_id}_{run_id}" if SAVE_SPEECH_AUDIO else None
                game = SpeechGameRunner(
                    client,
                    safe_delay_seconds=SAFE_DELAY_SECONDS,
                    answer_timeout_seconds=ANSWER_TIMEOUT_SECONDS,
                    logger=RunLogger(log_path),
                    transcriber=speech_transcribe,
                    audio_dir=audio_dir,
                    audio_fetch_delay_seconds=SPEECH_AUDIO_FETCH_DELAY_SECONDS,
                ).play(competition_id, strategy)
                summary = summarize_attempts(load_jsonl(log_path)) if log_path.exists() else {}
                live_run = {"competition_id": competition_id, "earned_amount": game.earned_amount, "log_path": str(log_path), "audio_dir": str(audio_dir) if audio_dir else None, "summary": summary}
                live_runs.append(live_run)
                print("Speech competition:", competition_id, CATEGORY_NAMES.get(competition_id), "Earned amount:", game.earned_amount, "Log path:", log_path)
                if audio_dir:
                    print("Audio dir:", audio_dir)
                print("Summary:", summary)
            result.update({
                "live_runs": live_runs,
                "log_paths": [run["log_path"] for run in live_runs],
                "earned_amount_total": sum((run.get("earned_amount") or 0) for run in live_runs),
            })
        run_results.append(result)
    except Exception as exc:
        print("ERROR", type(exc).__name__ + ":", exc)
        run_results.append({"plan": plan.get("plan"), "label": plan["label"], "error": f"{type(exc).__name__}: {exc}", "benchmark_ok": False})
    finally:
        if CLEAR_MEMORY_AFTER_EACH_MODEL:
            clear_model_memory(strategy, release_rag=True)
        del strategy

print("\nRun comparison:")
for item in run_results:
    print(item.get("label"), "earned_total=", item.get("earned_amount_total"), "benchmark_ok=", item.get("benchmark_ok"), "error=", item.get("error"))
run_results


## Results

Read the JSONL summaries from this execution.


In [13]:
from pathlib import Path
from polimillionaire.runner import load_jsonl, summarize_attempts

print("Benchmark results:")
for row in benchmark_results:
    print(row)

live_logs = []
for item in run_results:
    for log_path in item.get("log_paths", []):
        live_logs.append((item.get("label"), Path(log_path)))

if live_logs:
    print("\nSpeech live runs from this execution:")
    total_earned = 0
    for label, latest in live_logs:
        attempts = load_jsonl(latest)
        summary = summarize_attempts(attempts)
        earned = attempts[-1].get("result", {}).get("earned_amount", 0) if attempts else 0
        total_earned += earned or 0
        print(label)
        print(latest.name)
        print(summary, "earned:", earned)
    print("Total earned across displayed runs:", total_earned)
else:
    print("\nNo speech live game was run in this notebook execution.")
    old_logs = sorted((REPO_ROOT / "results" / "runs").glob("*.jsonl"), key=lambda path: path.stat().st_mtime)
    if old_logs:
        print("Most recent saved live log, not from this run:", old_logs[-1].name)


Benchmark results:
{'model': 'Single best - Gemma E2B two-agent quantized council + quantized judge', 'accuracy': 1.0, 'avg_seconds': 10.76, 'max_seconds': 18.05, 'fallbacks': 0, 'all_correct': True, 'wrong_questions': 0, 'by_category': {0: {'accuracy': 1.0, 'avg_seconds': 14.41, 'max_seconds': 17.33, 'fallbacks': 0, 'total': 4}, 1: {'accuracy': 1.0, 'avg_seconds': 15.14, 'max_seconds': 15.55, 'fallbacks': 0, 'total': 2}, 2: {'accuracy': 1.0, 'avg_seconds': 9.9, 'max_seconds': 16.0, 'fallbacks': 0, 'total': 3}, 3: {'accuracy': 1.0, 'avg_seconds': 3.82, 'max_seconds': 15.3, 'fallbacks': 0, 'total': 4}, 4: {'accuracy': 1.0, 'avg_seconds': 9.06, 'max_seconds': 16.09, 'fallbacks': 0, 'total': 3}, 5: {'accuracy': 1.0, 'avg_seconds': 16.78, 'max_seconds': 18.05, 'fallbacks': 0, 'total': 2}}}
{'model': 'Category best 0 entertainment - Gemma E2B two-agent quantized council + quantized judge', 'accuracy': 1.0, 'avg_seconds': 14.04, 'max_seconds': 15.98, 'fallbacks': 0, 'all_correct': True, 'wro